## Лабораторная работа № 1
## Выполнение разведочного анализа больших данных с использованием фреймворка Apache Spark

### Часть 1. Подготовка датасета Delivery ETA

Прочитаем датасет из HDFS, проверим структуру и типы признаков, выполним преобразования и сохраним результат в Apache Iceberg.

Объект наблюдения — потенциальный заказ в сервисе онлайн-доставки продуктов. Целевая переменная `delivery_eta_minutes` — время прибытия курьера в минутах. Описание признаков — в файле `description_data_set.md`. Исходный файл `delivery_eta.parquet` занимает около 10,3 ГиБ.

Ноутбук запускается на сервере с Spark 3.5 / Scala 2.12 и YARN. До первого запуска файл нужно загрузить из локальной папки удалённого сервера в HDFS. Выполните в **терминале сервера**, находясь в папке с ноутбуками:

```bash
# Проверяем HDFS и доступное место.
hdfs dfs -ls /
hdfs dfs -df -h

# Загружаем файл из локальной папки удалённого сервера в HDFS.
hdfs dfs -mkdir -p /user/$USER/datasets
hdfs dfs -copyFromLocal delivery_eta.parquet /user/$USER/datasets/

# Проверяем результат загрузки.
hdfs dfs -ls -h /user/$USER/datasets/delivery_eta.parquet
```

Эта загрузка выполняется один раз. Если файл уже находится в HDFS по указанному адресу, повторять команды не нужно. Вторая часть читает таблицу Iceberg, сохранённую этим ноутбуком.


Подключим необходимые библиотеки. Для преобразований используем PySpark DataFrame API.


In [138]:
import os  # Имя пользователя из окружения операционной системы.
from pyspark import SparkConf  # Настройки ресурсов Spark.
from pyspark.sql import SparkSession, DataFrame  # Сессия и распределённая таблица.
from pyspark.sql import functions as F  # Функции обработки столбцов.
import pandas as pd


In [139]:
from enum import Enum

class Version(Enum):
    prod = 0
    debug = 1

Укажем адрес предварительно загруженного файла в HDFS и имена базы и таблицы. Имя базы должно совпадать с указанным во второй части. При повторной записи содержимое таблицы заменяется.


In [140]:
# Имя текущего пользователя сервера для путей HDFS.
user_name = os.getenv("USER")

# Адрес датасета, заранее загруженного в HDFS.
path = f"hdfs:///user/{user_name}/datasets/delivery_eta.parquet"

# Имена базы и таблицы для подготовленных данных.

database_name = "delivery_eta_database"
table_name = "sobd_lab1_delivery_eta"
full_table_name = f"spark_catalog.{database_name}.{table_name}"


""" const """
COUNT_COL = 15
COUNT_ROW = 5
DEBUG_LIMIT_ROW=500_000
VERSION = Version.debug # todo поменять на прод. уменьшает размер df
NAN = "NaN"
NULL = "Null"
PERSENT_NAN = "Nan %"
PERSENT_NULL = "Null %"


Настроим Spark для выполнения на YARN: два исполнителя по 8 ядер и 12 ГБ памяти. Подключим Iceberg и зададим пользовательское хранилище таблиц в HDFS.


In [141]:
def create_spark_configuration() -> SparkConf:
    
    conf = SparkConf()
    conf.setAppName("lab 1 Delivery ETA")
   
    conf.setMaster("yarn")
    conf.set("spark.submit.deployMode", "client")
    conf.set("spark.executor.memory", "12g")
    # Каждому исполнителю выделяем 8 процессорных ядер.
    conf.set("spark.executor.cores", "8") # исполнителю - 8 ядер
    conf.set("spark.executor.instances", "2") #по 2 сущности исполнителя
    conf.set("spark.driver.memory", "4g")
    conf.set("spark.driver.cores", "2")
    
    conf.set("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0")
    conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    conf.set("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkCatalog")
    conf.set("spark.sql.catalog.spark_catalog.type", "hadoop")
    conf.set("spark.sql.catalog.spark_catalog.warehouse", f"hdfs:///user/{user_name}/warehouse")
    conf.set("spark.sql.catalog.spark_catalog.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
    return conf

conf = create_spark_configuration()


Создадим сессию Spark. Если в ядре уже работает сессия с другими настройками, сначала остановите её или перезапустите ядро.


In [142]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark


26/11/02 05:06:22 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


### Чтение данных из HDFS

Прочитаем предварительно загруженный файл в датафрейм Spark. Parquet хранит названия и типы столбцов, поэтому отдельно задавать схему не требуется.

Число строк и диапазон целевой переменной вычислим ниже на загруженных данных. Короткий файл `delivery_eta.csv` — только образец строк, расчёты выполняем по полному Parquet.


In [144]:
from collections import defaultdict

BIN = "bin"
NUM = "num"
CAT = "cat"

def filrer_col(cols: list[str])-> defaultdict[str, list[str]]:
    names: defaultdict[str, list[str]] = defaultdict(list[str]) 
    for col in cols:
        if col.startswith("num_"):
            key = NUM
        elif col.startswith("bin_"):
            key = BIN
        elif col.startswith("cat_"):
            key = CAT
        else:
            key = "other"
        names[key].append(col)
         
    return names

_tmp_col: list[str] = list(spark.read.parquet(path).limit(1).schema.names)

COLUMNS = filrer_col(_tmp_col)

if VERSION is Version.debug:
    COLUMNS.update({c: COLUMNS[c][:COUNT_COL] for c in (BIN, NUM, CAT)})

ALL_COLUMNS = COLUMNS[NUM] + COLUMNS[BIN] + COLUMNS[CAT] + COLUMNS["other"]

In [145]:
print(VERSION)
df = spark.read.parquet(path).select(*ALL_COLUMNS)
df = df.limit(DEBUG_LIMIT_ROW) if VERSION is Version.debug else df

TYPES = dict(df.dtypes)
TOTAL_ROWS = df.count()   
df.printSchema()

Version.debug


root
 |-- num_0: float (nullable = true)
 |-- num_1: float (nullable = true)
 |-- num_2: float (nullable = true)
 |-- num_3: float (nullable = true)
 |-- num_4: float (nullable = true)
 |-- num_5: float (nullable = true)
 |-- num_6: float (nullable = true)
 |-- num_7: float (nullable = true)
 |-- num_8: float (nullable = true)
 |-- num_9: float (nullable = true)
 |-- num_10: float (nullable = true)
 |-- num_11: float (nullable = true)
 |-- num_12: float (nullable = true)
 |-- num_13: float (nullable = true)
 |-- num_14: float (nullable = true)
 |-- bin_0: float (nullable = true)
 |-- cat_0: long (nullable = true)
 |-- cat_1: long (nullable = true)
 |-- cat_2: long (nullable = true)
 |-- timestamp: timestamp_ntz (nullable = true)
 |-- delivery_eta_minutes: float (nullable = true)



In [146]:

for key, value in COLUMNS.items():
    print(f"key: {key}, value: {len(value)}")

df.limit(COUNT_ROW).toPandas().head(COUNT_ROW)

key: num, value: 15
key: bin, value: 1
key: cat, value: 3
key: other, value: 2


,num_0,num_1,num_2,num_3,num_4,num_5,num_6,num_7,num_8,num_9,...,num_11,num_12,num_13,num_14,bin_0,cat_0,cat_1,cat_2,timestamp,delivery_eta_minutes
0,-1.393848,-0.624145,-0.249721,0.280911,-1.394363,-0.615658,-0.249482,0.282054,-1.778632,-1.008543,...,2.003146,2.326520,-0.890382,-0.288336,0.0,1,1,1,2023-11-25 19:01:37,1.883333
1,0.344819,1.114033,1.121060,0.098013,0.359881,1.122287,1.120645,0.094761,-0.252971,1.788387,...,0.946948,-0.859450,-0.276501,0.230411,1.0,2,2,2,2023-12-12 12:12:12,6.433333
2,-1.641144,-1.612867,-0.774833,1.233962,-1.632285,-1.614242,-0.767196,1.242014,-0.655411,-0.460893,...,0.513749,-1.224724,-0.504805,0.015619,1.0,3,3,3,2023-12-14 11:07:59,5.066667
3,-0.637411,0.481779,0.718462,-0.265284,-0.576190,0.498744,0.730634,-0.360672,1.106778,1.972344,...,0.377279,-0.322041,0.173909,0.738644,1.0,4,4,4,2023-10-30 09:25:34,7.716667
4,0.053135,0.394525,0.267884,-1.123415,0.084245,0.449845,0.345423,-1.174407,0.977586,-0.578083,...,1.274747,-0.472398,0.034169,0.566346,0.0,5,5,5,2023-12-17 16:22:38,17.316668


### Содержание датасета и назначение столбцов

Каждая строка описывает потенциальный заказ в сервисе онлайн-доставки продуктов. Задача — прогнозировать, через сколько минут прибудет курьер. Признаки обезличены: они агрегируют доступность курьеров, гео- и навигационные оценки, историю предыдущих доставок и характеристики времени суток.

| Название столбца | Расшифровка |
|---|---|
| `num_0` … `num_220` | 221 обезличенный стандартизованный числовой признак. Значения обычно около нуля, во многих столбцах есть пропуски |
| `bin_0` | Обезличенный бинарный признак со значениями 0 и 1. Точная семантика не раскрыта |
| `cat_0` | Первый обезличенный категориальный признак. Значения — числовые коды категорий |
| `cat_1` | Второй обезличенный категориальный признак. Значения — числовые коды категорий |
| `cat_2` | Третий обезличенный категориальный признак. Значения — числовые коды категорий |
| `timestamp` | Дата и время формирования заказа. Используется для анализа по времени и хронологического разбиения данных |
| `delivery_eta_minutes` | Время прибытия курьера в минутах. Целевая переменная для прогнозирования |

Всего в исходном файле **227 столбцов**: 225 входных признаков, временная метка и целевая переменная.

Числовые коды `cat_*` обозначают категории, поэтому не трактуем их как количественные величины. Среднее арифметическое по коду зоны или типа заказа предметного смысла не имеет.


### Первичные метрики

на числовые параметры

| Название столбца | Расшифровка |
|---|---|
|count|Количество строк|
|mean|Среднее значение|
|stddev|Стандартное отклонение, насколько значения в среднем разбросаны вокруг среднего|
|min|Минимальное значение|
|25%|Персентиль 25%|
|50%|Персентиль 50%|
|75%|Персентиль 75%|
|max|Максимальное занчение|



In [147]:
n_cores = spark.sparkContext.defaultParallelism
part_number = df.rdd.getNumPartitions()
count_reparatiton = n_cores * max(part_number // part_number, 2)
print(f"Количество ядер - {n_cores}")
print(f"Количество партиций - {part_number}")
print(f"count repartition - {count_reparatiton}")

tmp_df = df.repartition(count_reparatiton) if part_number < n_cores else df

summary = tmp_df.summary() if VERSION is Version.debug else tmp_df.describe()
print("Сводная таблица")
summary.toPandas().set_index("summary").T.head(len(df.columns))

Количество ядер - 16
Количество партиций - 1
count repartition - 32
Сводная таблица


summary,count,mean,stddev,min,25%,50%,75%,max
num_0,500000,1.6642702360078693E-4,1.0107468717044377,-4.8455844,-0.67415226,0.0029273294,0.68316925,4.648853
num_1,500000,-0.0035724769941703417,1.0103759281287479,-4.741838,-0.6849896,-0.0018818531,0.67475235,4.2659206
num_2,500000,-0.004011384268064517,1.0096431668615382,-4.627289,-0.69112104,-0.006272881,0.67527753,4.585457
num_3,500000,0.009653925252219663,1.008367077668923,-3.089935,-0.6658523,0.013800685,0.6847631,5.1993375
num_4,500000,0.001532741366266353,1.0055320838015311,-4.1349034,-0.6745977,0.005367909,0.6834236,5.1993375
num_5,500000,-0.00514347969402757,1.0073005719805515,-3.8607757,-0.689113,-0.0026681826,0.67860806,5.1993375
num_6,500000,-0.0022784249754624324,1.0031093122381993,-5.1993375,-0.68670493,-0.0029772315,0.678616,4.076813
num_7,500000,0.01004300791244139,1.0015336648405337,-5.08913,-0.6648397,0.015490358,0.68346095,4.4328785
num_8,500000,-0.004197412172269771,0.9953174918078788,-3.2124383,-0.6816258,-0.0028861677,0.67267317,3.8383608
num_9,500000,0.003980738293453731,0.9995659104865988,-3.1428502,-0.67438793,0.0010029672,0.6778497,4.7507577


### `bin` признаки

проверяем на Nan & Null

Переделаем `bin` признаки в булевый параметр

In [148]:
def check_miss_float(df: DataFrame) -> pd.DataFrame:
    col = list(df.columns)

    if len(col) == 0: 
        return pd.DataFrame(
            columns=[NAN, NULL, PERSENT_NAN, PERSENT_NULL]
        )

    if any(TYPES[c] not in {"double", "float"} for c in col):
        raise ValueError("Use only `double`, `float` columns")

    # 1) выравниваем партиции ТОЛЬКО если их мало
    n_cores = df.sparkSession.sparkContext.defaultParallelism
    if df.rdd.getNumPartitions() < n_cores:
        df = df.repartition(n_cores * 2)

    # 2) все проверки в один agg — ОДИН скан вместо двух
    exprs = []
    for c in col:
        exprs.extend([
            F.sum(F.col(c).isNull().cast("int")).alias(f"{c}__nulls"),
            F.sum(F.isnan(F.col(c)).cast("int")).alias(f"{c}__nans"),
        ])

    row = df.agg(*exprs).collect()[0].asDict()

    miss = pd.DataFrame(
        {
            NAN:  [row.get(f"{c}__nans", 0) for c in col],
            NULL: [row.get(f"{c}__nulls", 0) for c in col],
        },
        index=col,
    )

    miss[PERSENT_NAN] = miss[NAN] / TOTAL_ROWS * 100
    miss[PERSENT_NULL] = miss[NULL] / TOTAL_ROWS * 100
    return miss

Проверка на Null & NaN

In [149]:
miss = check_miss_float(df.select(*COLUMNS[BIN]))
miss.head(len(miss))


,NaN,Null,Nan %,Null %
bin_0,0,0,0.0,0.0


Преобразование `bin`

In [150]:
df = df.withColumns({c: F.col(c).cast("Boolean") for c in COLUMNS[BIN]})
df.select(*COLUMNS[BIN]).printSchema()

root
 |-- bin_0: boolean (nullable = true)



### Timestamp

In [151]:
TIMESTAMP = "timestamp"
print(TYPES[TIMESTAMP])

timestamp_ntz


Смотрим данные

In [152]:
timestamp_col = df.select(TIMESTAMP)
timestamp_col.limit(COUNT_ROW).toPandas().head(COUNT_ROW)

,timestamp
0,2023-11-25 19:01:37
1,2023-12-12 12:12:12
2,2023-12-14 11:07:59
3,2023-10-30 09:25:34
4,2023-12-17 16:22:38


Создаём сводную таблицу

In [153]:
rng = df.agg(
    F.min(TIMESTAMP).alias("min_ts"),
    F.max(TIMESTAMP).alias("max_ts"),
    F.count(TIMESTAMP).alias("cnt"),
    F.sum(F.col(TIMESTAMP).isNull().cast("int")).alias(NULL)
).toPandas().T
rng.rename(columns={0:"Value"}).head(len(rng))

,Value
min_ts,2023-10-20 01:01:32
max_ts,2024-01-07 22:48:50
cnt,500000
Null,0


### Delivery eta minutes

In [154]:
DEM = "delivery_eta_minutes"
print(TYPES[DEM])

float


In [155]:
dem_col = df.select(DEM)
dem_col.limit(COUNT_ROW).toPandas().head(COUNT_ROW)

,delivery_eta_minutes
0,1.883333
1,6.433333
2,5.066667
3,7.716667
4,17.316668


In [156]:
miss = check_miss_float(df.select(DEM))
miss.head(len(miss))

,NaN,Null,Nan %,Null %
delivery_eta_minutes,0,0,0.0,0.0


### `Cat` обезлеченные категории

In [157]:
for c in COLUMNS[CAT]:
    print(f"col: {c}, type: {TYPES[c]}")

col: cat_0, type: bigint
col: cat_1, type: bigint
col: cat_2, type: bigint


Check `NUll`

In [158]:
checks_null = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in COLUMNS[CAT]]

n_cores = df.sparkSession.sparkContext.defaultParallelism
tmp_df = df.repartition(n_cores * 2) if df.rdd.getNumPartitions() < n_cores else df
       
df = df.repartition(n_cores * 2)
miss = df.select(*checks_null).toPandas().T.rename(columns={0: NULL})
miss[PERSENT_NULL] = miss[NULL] / TOTAL_ROWS * 100
miss.head(len(miss))

,Null,Null %
cat_0,0,0.0
cat_1,0,0.0
cat_2,0,0.0


Дубликаты

In [159]:
cond1 = F.col("cat_0").eqNullSafe(F.col("cat_1"))
cond2 = F.col("cat_1").eqNullSafe(F.col("cat_2"))
cond3 = F.col("cat_0").eqNullSafe(F.col("cat_2"))

print(TOTAL_ROWS)

pairs = {
    "cat_0 == cat_1": cond1,
    "cat_1 == cat_2": cond2,
    "cat_0 == cat_2": cond3,
    "cat_0 == cat_1 | cat_0 == cat_2": cond1 | cond3,
    "cat_0 == cat_1 | cat_1 == cat_2": cond1 | cond2,
    "cat_0 == cat_2 | cat_1 == cat_2": cond3 | cond2,
    "any":   cond1 | cond2 | cond3,
}

df.select(*[
    F.sum(expr.cast("int")).alias(name) 
    for name, expr in pairs.items()
]).toPandas().T.rename(columns={0: "duplicate"}).head(len(pairs))

500000


,duplicate
cat_0 == cat_1,129833
cat_1 == cat_2,500000
cat_0 == cat_2,129833
cat_0 == cat_1 | cat_0 == cat_2,129833
cat_0 == cat_1 | cat_1 == cat_2,500000
cat_0 == cat_2 | cat_1 == cat_2,500000
any,500000


Удаляем дублирующие строки

In [160]:
COLUMNS[CAT] = ["cat_0", "cat_1"]
df = df.drop("cat_2")

### `Num` показатели

In [161]:
for c in COLUMNS[NUM]:
    print(f"col: {c}, type: {TYPES[c]}")

col: num_0, type: float
col: num_1, type: float
col: num_2, type: float
col: num_3, type: float
col: num_4, type: float
col: num_5, type: float
col: num_6, type: float
col: num_7, type: float
col: num_8, type: float
col: num_9, type: float
col: num_10, type: float
col: num_11, type: float
col: num_12, type: float
col: num_13, type: float
col: num_14, type: float


In [162]:
miss = check_miss_float(df.select(*COLUMNS[NUM]))
miss.head(len(miss))


,NaN,Null,Nan %,Null %
num_0,0,0,0.0000,0.0
num_1,0,0,0.0000,0.0
num_2,0,0,0.0000,0.0
num_3,0,0,0.0000,0.0
num_4,0,0,0.0000,0.0
num_5,0,0,0.0000,0.0
num_6,0,0,0.0000,0.0
num_7,0,0,0.0000,0.0
num_8,0,0,0.0000,0.0
num_9,0,0,0.0000,0.0


Заполнение Nan на 0.0 тк их не много

In [163]:
miss_cond = (miss[NAN] > 0) | (miss[NULL] > 0)
miss_col = list(miss.loc[miss_cond].index)

df = df.na.fill({c: 0.0 for c in miss_col})

miss_df = check_miss_float(df.select(*miss_col))
miss_df.head(len(miss_df))

,NaN,Null,Nan %,Null %
num_10,0,0,0.0,0.0
num_11,0,0,0.0,0.0


### Проверим все что можем (


In [164]:
print(f"Количество строк: {TOTAL_ROWS}")

Количество строк: 500000


In [165]:
# select() выбирает результат расчёта.
# first() получает единственную строку результата, [0] — значение в ней.
first_time = df.select(F.min(TIMESTAMP)).first()[0]
last_time = df.select(F.max(TIMESTAMP)).first()[0]
print(f"Период наблюдений: {first_time} — {last_time}")

Период наблюдений: 2023-10-20 01:01:32 — 2024-01-07 22:52:00


In [166]:
min_eta = df.select(F.min(DEM)).first()[0]
max_eta = df.select(F.max(DEM)).first()[0]
print(f"Минимальное время прибытия: {min_eta}")
print(f"Максимальное время прибытия: {max_eta}")

Минимальное время прибытия: 0.0
Максимальное время прибытия: 129.28334045410156


In [167]:
# Сначала отбираем нужные строки, затем считаем их.
negative_time_rows = df.filter(F.col(DEM) < 1)
negative_time_count = negative_time_rows.count()
print(f"Строк с временем прибытия или мгновенным временем прибытия: {negative_time_count}")

Строк с временем прибытия или мгновенным временем прибытия: 11442


Проверка на закономерность на время меньше 1 минуты

In [168]:
# проверка моей гипотезы
# переводим время в секунды и проверяем заказы меньше 30 секунд
# Мировой рекорд Усэйна Болта на дистанции 100 метров - 9,58 секунды
prop = df.filter((F.col(DEM) * 60) <= 60).select(
    (F.col(DEM) * 60).alias("seconds"),
    *[F.col(c) for c in COLUMNS[BIN]],
    *[F.col(c) for c in COLUMNS[CAT]],
)
prop.limit(20).toPandas().head(20)

,seconds,bin_0,cat_0,cat_1
0,52.0,False,124,125
1,4.0,True,251,260
2,41.0,True,36,36
3,8.0,True,103,131
4,7.0,False,437,462
5,56.0,False,70,70
6,14.0,False,336,353
7,7.0,True,426,449
8,4.0,False,352,369
9,55.0,True,310,323


Удалим Шум - не бывает таких случаев;
нельзя по данным понять - отмененан она или нет.
а корреляцию не приходит на ум

In [169]:
df = df.filter(F.col(DEM) >= 1)
print(df.count())

488558


In [170]:
missing_target_rows = df.filter(F.col(DEM).isNull())
missing_target_count = missing_target_rows.count()
print(f"Пропусков времени прибытия: {missing_target_count}")

Пропусков времени прибытия: 0


In [171]:
missing_timestamp_rows = df.filter(F.col(TIMESTAMP).isNull())
missing_timestamp_count = missing_timestamp_rows.count()
print(f"Пропусков временной метки: {missing_timestamp_count}")

Пропусков временной метки: 0


In [ ]:
# Передаём SQL-команду Spark: создаём базу, если её ещё нет; существующая база сохраняется.
db_short = database_name.split(".")[-1] 
print(db_short, database_name, sep='\n')

spark.sql("SHOW DATABASES IN spark_catalog").show(truncate=False)
spark.sql("SHOW NAMESPACES IN spark_catalog").show(truncate=False)
spark.sql(f"USE spark_catalog.{database_name}")
print(spark.catalog.currentDatabase())     # delivery_eta_database
print(spark.catalog.listTables(), sep='\n')        # sobd_lab1_delivery_eta, ..._processed

# spark.sql(f"CREATE DATABASE IF NOT EXISTS spark_catalog.{database_name}")
# Выбираем базу текущей: короткие имена таблиц будут искаться в ней.
# spark.catalog.setCurrentDatabase(database_name)

delivery_eta_database
delivery_eta_database
+----------------------+
|namespace             |
+----------------------+
|Winter_Ivan_database  |
|albadwi_database      |
|alieva_SAPR_database  |
|alieva_database       |
|cooking_time_database |
|delivery_eta_database |
|dolgova_database      |
|drynkin_database      |
|elistratov_database   |
|grebenkin_database    |
|ivanov_database       |
|ivanov_evm_12_database|
|krutov_database       |
|orlov_database        |
|pavlov_database       |
|varlamov_database     |
+----------------------+

+----------------------+
|namespace             |
+----------------------+
|Winter_Ivan_database  |
|albadwi_database      |
|alieva_SAPR_database  |
|alieva_database       |
|cooking_time_database |
|delivery_eta_database |
|dolgova_database      |
|drynkin_database      |
|elistratov_database   |
|grebenkin_database    |
|ivanov_database       |
|ivanov_evm_12_database|
|krutov_database       |
|orlov_database        |
|pavlov_database       |
|varl

Запишем таблицу.


In [183]:
# writeTo() указывает, в какую таблицу записать df; using() задаёт формат Iceberg.
# createOrReplace() создаёт таблицу или заменяет существующую. Здесь реально запускаются вычисления и запись.
df.writeTo(full_table_name).using("iceberg").createOrReplace()


Проверим наличие таблицы и прочитаем небольшой фрагмент уже из Iceberg.


In [184]:
# Показываем список таблиц базы и проверяем, что наша таблица появилась.
spark.sql(f"SHOW TABLES IN spark_catalog.{database_name}").show(truncate=False)
# Читаем сохранённую таблицу по её полному имени в новый DataFrame.
saved_df = spark.table(full_table_name)

saved_df.select(df.columns).limit(COUNT_ROW).toPandas().head(COUNT_ROW)



+---------------------+--------------------------------+-----------+
|namespace            |tableName                       |isTemporary|
+---------------------+--------------------------------+-----------+
|delivery_eta_database|sobd_lab1_delivery_eta          |false      |
|delivery_eta_database|sobd_lab1_delivery_eta_processed|false      |
+---------------------+--------------------------------+-----------+



,num_0,num_1,num_2,num_3,num_4,num_5,num_6,num_7,num_8,num_9,num_10,num_11,num_12,num_13,num_14,bin_0,cat_0,cat_1,timestamp,delivery_eta_minutes
0,-1.088722,-0.341493,-0.155776,-0.385726,-1.074384,-0.360425,-0.147070,-0.382587,-1.704141,-0.940608,-1.895259,0.103376,0.219570,0.780213,2.144146,True,47,47,2023-11-30 13:44:43,1.700000
1,0.667406,-1.140046,-1.204921,0.478025,0.669801,-1.129089,-1.213381,0.488226,1.537240,1.412201,0.910873,-2.373293,0.102382,0.631788,1.535936,True,118,119,2023-11-06 10:37:26,24.966667
2,-0.013801,0.245840,0.124523,-0.963423,-0.021146,0.285046,0.162959,-1.080752,1.019727,-0.379755,1.475272,0.809210,-0.476754,0.030167,0.561714,False,79,262,2023-11-12 19:28:43,5.166667
3,0.611987,-1.315866,-1.424852,0.475214,0.616267,-1.340054,-1.421164,0.479637,-0.397395,1.187402,-0.132116,1.097071,-0.980420,-0.360029,0.150172,True,193,199,2023-12-01 19:16:37,3.283333
4,0.462411,0.680018,0.392492,-0.111890,0.471102,0.712398,0.428240,-0.095489,-0.222701,-1.017052,-0.204992,0.411359,0.537285,1.283831,-1.273090,True,363,381,2023-12-08 12:48:05,4.733333


После завершения работы остановим сессию Spark.


In [185]:
spark.stop()